# FPGA DSP Design & Verification with LLMs

Put simply, LLMs have gotten crazy good recently, going beyond typical software coding tasks, to even tackling HDL design. As those with experience in digital design know, this domain is not as simple to "code" as software: you're really trying to coerce a chain of- often proprietary- EDA tools to map a desired function to a series of wires and hardware-specific primitives. Nonetheless, and even with much less open-source/high-quality training data, recent models have made leaps and bounds progress.

As an example, we want to design a canonical [Numerically Controlled Oscillator](https://john-gentile.com/kb/dsp/NCO_DDS.html) for use in a DSP FPGA design. The following HDL was generated with [Claude Opus 4.6](https://claude.ai/) with the below simple prompt and no other project nor system context:
> Create a simple Numerically Controlled Oscillator (NCO) for use in an FPGA in modern/idiomatic SystemVerilog

In [ ]:
from IPython.display import Markdown as md

In [ ]:
with open('./FPGA_NCO_files/nco.sv', 'r') as f:
    f_contents = f.read()

md(f"\n```verilog\n{f_contents}\n```")

^ this looks pretty good for one-shot output! I mean, even with "simple" in the prompt, it went beyond the basic "phase accumulator -> LUT" design and right to implementing:
- A quarter-wave LUT to save on space with built-in $\sin$ value generation within the code (no external memory loading required)!
- Built-in LFSR for phase dithering to increase SFDR.
- Assertions to check for proper instantiation and usage.

But now we have to verify the design- let's switch to [Claude Code](https://code.claude.com/docs/en/overview) to give the context of our `nco.sv` HDL file and let it generate a testbench and `Makefile` for use by [Verilator](https://www.veripool.org/verilator/):
> @nco.sv is a simple Numerically Controlled Oscillator in SystemVerilog. We want to write a simple unit test to verify basic output- write a simple SystemVerilog testbench as well as Makefile targeting Verilator simulator (Verilator is already installed on the system).

Claude goes right to work making the following testbench, but not without using it's Bash tool first to find the current Verilator version installed (`$ verilator --version`) to make sure the testbench matches supported conventions (brilliant!):

In [ ]:
with open('./FPGA_NCO_files/nco_tb.sv', 'r') as f:
    f_contents = f.read()

md(f"\n```verilog\n{f_contents}\n```")

Now Claude knows it needs to generate a Makefile to launch the simulation and check outputs:

In [ ]:
with open('./FPGA_NCO_files/Makefile', 'r') as f:
    f_contents = f.read()

md(f"\n```makefile\n{f_contents}\n```")

It then runs the Makefile, sees an error related to `IMPLICITSTATIC` lifetime, adjusts the Makefile to ignore that flag in the one variable, and re-runs to show a successul testbench execution:
```
- V e r i l a t i o n   R e p o r t: Verilator 5.046 2026-02-28 rev vUNKNOWN-built20260228
- Verilator: Built from 0.061 MB sources in 3 modules, into 0.093 MB in 9 C++ files needing 0.000 MB
- Verilator: Walltime 2.420 s (elab=0.006, cvt=0.011, bld=2.392); cpu 0.021 s on 12 threads; allocated 11.875 MB
./obj_dir/Vnco_tb
[TEST 1] Reset behavior
[TEST 2] Valid strobe latency
[TEST 3] FCW=0 constant output
[TEST 4] Bounded output with active tone
  PASS: all samples within [-2047, 2047]
[TEST 5] Quadrature check (cos leads sin by ~90 deg)
  PASS: cos > sin near phase 0 (sin=783 cos=1871)

=== ALL TESTS PASSED ===
- nco_tb.sv:156: Verilog $finish
```

Let's go one step further- let's have Claude use it's Python skills to generate a [cocotb](https://www.cocotb.org/) testbench to numerically verify the NCO output:
> Now that we have a basic unit testbench in @nco_tb.sv let's create a `cocotb` Python verification file that we can run to plot the output spectrum of @nco.sv and measure the SFDR.

This gives us another [Makefile to launch the cocotb sim](./FPGA_NCO_files/Makefile.cocotb) and then the following Python testbench:

In [ ]:
with open('./FPGA_NCO_files/test_nco.py', 'r') as f:
    f_contents = f.read()

md(f"\n```python\n{f_contents}\n```")

This gives the following plot and SFDR measurement output:
![SFDR plot](./FPGA_NCO_files/nco_spectrum.png)

## Summary

In the end, this isn't a _perfect_ solution- there's some quirks in the dither logic and we need to pass through a tool like [Vivado's](https://www.amd.com/en/products/software/adaptive-socs-and-fpgas/vivado.html) synthesis flow to know if we can map this to a target FPGA- but this is a massive and quick start to a DSP FPGA design that we can iterate on further. And again, this was done with no system or other context- the true power of LLMs in this domain, and moreover agentic tools like Claude Code, is:
* Feeding the rest of your codebase and documentation as context (especially useful in complex DSP projects where we have documentation like waveform specs and ICDs, as well as modeling code, like Matlab).
* Specifying HDL testbench coverage, coding-style, linting, etc. requirements as part of system context (like in Claude Code's `CLAUDE.md` instruction file)- LLMs are particularly useful at the mundane tasks of documenting blocks, adding tests, formatting, etc.
* Hooking up other tools, like Vivado `tcl` steps, as [Model Context Protocol (MCP)](https://en.wikipedia.org/wiki/Model_Context_Protocol) tools available to be called by the LLM.